In [2]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/Customer_support_chatbot'
!pip install -q datasets transformers accelerate evaluate

MessageError: Error: credential propagation was unsuccessful

In [ ]:
from datasets import load_dataset
sentiment_ds = load_dataset("dair-ai/emotion")
print(sentiment_ds)

label_names = sentiment_ds["train"].features["label"].names
print(label_names)

map 6 emotions to routing buckets

In [ ]:
sentiment_to_buckets = {
    'sadness': 'negative',
    'anger': 'negative',
    'fear': 'negative',
    'joy': 'positive',
    'love': 'positive',
    'surprise': 'neutral',
}

bucket_to_id = {
    'negative': 0,
    'neutral': 1,
    'positive': 2,
}

def remap_labels(examples):
  original_label_name = label_names[examples['label']]
  bucket_name = sentiment_to_buckets[original_label_name]
  examples['bucket'] = bucket_to_id[bucket_name]
  return examples

sentiment_ds = sentiment_ds.map(remap_labels)
print(sentiment_ds)

text tokenization

In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
  return tokenizer(batch['text'], truncation=True,
                   padding='max_length')

tokenized_ds = sentiment_ds.map(tokenize_fn, batched=True)
print(tokenized_ds)

Set up the model and Trainer

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir=f"{PROJECT_DIR}/sentiment_checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds['train'].rename_column('bucket', 'labels'),
    eval_dataset=tokenized_ds['validation'].rename_column('bucket', 'labels'),
    compute_metrics=compute_metrics,
)

In [ ]:
# TRAINING:
trainer.train()

In [ ]:
test_results = trainer.evaluate(tokenized_ds['test'].rename_column('bucket', 'labels'))
print(test_results)

In [ ]:
import os
save_path = f"{PROJECT_DIR}/models/sentiment_model"
os.makedirs(save_path, exist_ok=True)
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

sanity test

In [ ]:
from transformers import pipeline

sentiment_pipe = pipeline("text-classification", model=save_path, tokenizer=save_path)
id_to_bucket = {0: 'negative', 1: 'neutral', 2: 'positive'}

result = sentiment_pipe("This is the third time my order is late, I'm so frustrated!")
print(result)  # will show LABEL_0/1/2 — map using id_to_bucket